In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 40)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")


def section(title):
    """Print a section header for readable notebook output."""
    line = "=" * 70
    print(f"\n{line}\n{title}\n{line}")


DATA_PATH = Path("data/processed/f1_driver_race_dataset_2023_2025.csv")
CLEAN_DATA_PATH = Path("data/processed/f1_driver_race_dataset_cleaned.csv")

OUTPUT_DIR = Path("outputs")
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

def numericize(df, cols):
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def cohen_d(x, y):
    """Cohen's d for (x - y). Positive d means x > y on average."""
    x = pd.Series(x).dropna()
    y = pd.Series(y).dropna()

    if len(x) < 2 or len(y) < 2:
        return np.nan

    x_var = x.var(ddof=1)
    y_var = y.var(ddof=1)

    pooled_std = np.sqrt(
        ((len(x) - 1) * x_var + (len(y) - 1) * y_var) / (len(x) + len(y) - 2)
    )
    if pooled_std == 0:
        return np.nan

    return (x.mean() - y.mean()) / pooled_std


def save_fig(filename):
    plt.tight_layout()
    plt.savefig(FIG_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()

section("STEP 1 | Loading dataset")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}\n"
        "Please run 01_data_collection.ipynb first."
    )

df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset : {DATA_PATH}")
print(f"Shape          : {df.shape[0]} rows x {df.shape[1]} cols")

print("\nFirst 10 rows:")
display(df.head(10))


section("STEP 2 | Initial inspection")

print("All columns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values (top 25):")
missing_initial = (
    df.isna().sum().sort_values(ascending=False).head(25).to_frame("missing_count")
)
missing_initial["missing_pct"] = (
    missing_initial["missing_count"] / len(df) * 100
).round(2)
display(missing_initial)

dup_check = (
    df.groupby(["session_key", "driver_number"]).size().reset_index(name="n")
)
problem_dups = dup_check[dup_check["n"] > 1]
print(f"\nDuplicate driver-race keys: {len(problem_dups)}")
if len(problem_dups) > 0:
    display(problem_dups.head(20))


section("STEP 3 | Cleaning dataset")

df_clean = df.copy()

numeric_cols = [
    "year", "driver_number",
    "grid_position", "finishing_position",
    "positions_gained", "abs_position_change",
    "quali_lap_duration", "duration", "number_of_laps",
    "n_pit_stops", "first_pit_lap", "avg_pit_lap",
    "avg_pit_duration", "avg_pit_lane_duration",
    "n_stints", "avg_stint_length", "max_stint_length",
    "n_compounds_used", "avg_tyre_age_at_start",
    "latitude", "longitude", "weather_hours",
    "temperature_mean", "humidity_mean",
    "precipitation_sum", "precipitation_max",
    "wind_speed_mean", "weather_code_mode",
]
df_clean = numericize(df_clean, numeric_cols)

for col in ["date_start", "date_end"]:
    if col in df_clean.columns:
        df_clean[col] = pd.to_datetime(df_clean[col], errors="coerce")

for col in ["dnf", "dns", "dsq"]:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].map(
            lambda x: False if pd.isna(x) else bool(x)
        ).astype(bool)

if "weather_class" in df_clean.columns:
    df_clean["weather_class"] = (
        df_clean["weather_class"]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace({"nan": np.nan})
    )

before_dedup = len(df_clean)
df_clean = df_clean.drop_duplicates(
    subset=["session_key", "driver_number"], keep="first"
)
after_dedup = len(df_clean)
if before_dedup != after_dedup:
    print(f"WARNING: dropped {before_dedup - after_dedup} duplicate driver-race rows.")

df_clean = df_clean[df_clean["driver_number"].notna()].copy()

if "dns" in df_clean.columns:
    df_clean = df_clean[~df_clean["dns"]].copy()
if "dsq" in df_clean.columns:
    df_clean = df_clean[~df_clean["dsq"]].copy()

df_clean["positions_gained"] = (
    df_clean["grid_position"] - df_clean["finishing_position"]
)
df_clean["abs_position_change"] = (
    df_clean["grid_position"] - df_clean["finishing_position"]
).abs()

if "n_pit_stops" in df_clean.columns:
    df_clean["n_pit_stops"] = df_clean["n_pit_stops"].fillna(0).astype(int)

for col in ["n_stints", "n_compounds_used"]:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(0)


analysis_df = df_clean.copy()

dropped_dnf = 0
if "dnf" in analysis_df.columns:
    dropped_dnf = int(analysis_df["dnf"].sum())
    analysis_df = analysis_df[~analysis_df["dnf"]].copy()

before_pos_filter = len(analysis_df)
analysis_df = analysis_df[
    analysis_df["grid_position"].notna()
    & analysis_df["finishing_position"].notna()
].copy()
dropped_missing_pos = before_pos_filter - len(analysis_df)

analysis_df = analysis_df[
    (analysis_df["grid_position"] > 0) & (analysis_df["finishing_position"] > 0)
].copy()

analysis_df["positions_gained"] = (
    analysis_df["grid_position"] - analysis_df["finishing_position"]
)
analysis_df["abs_position_change"] = (
    analysis_df["grid_position"] - analysis_df["finishing_position"]
).abs()

if "weather_class" in analysis_df.columns:
    analysis_df = analysis_df[
        analysis_df["weather_class"].isin(["dry", "mixed", "wet"])
    ].copy()
    analysis_df["weather_group"] = np.where(
        analysis_df["weather_class"] == "dry", "dry", "wet_or_mixed"
    )

df_clean.to_csv(CLEAN_DATA_PATH, index=False)
analysis_df.to_csv(TABLE_DIR / "analysis_dataset_used_for_tests.csv", index=False)

filter_audit = pd.DataFrame({
    "step": [
        "raw rows",
        "after dedup",
        "after DNS/DSQ drop",
        "dropped as DNF",
        "dropped due to missing grid/finish",
        "final analysis rows",
    ],
    "rows": [
        len(df),
        after_dedup,
        len(df_clean),
        dropped_dnf,
        dropped_missing_pos,
        len(analysis_df),
    ],
})
display(filter_audit)

print(f"\nCleaned dataset shape   : {df_clean.shape}")
print(f"Analysis dataset shape  : {analysis_df.shape}")
print(f"Cleaned dataset saved   : {CLEAN_DATA_PATH}")
print(f"Analysis dataset saved  : {TABLE_DIR / 'analysis_dataset_used_for_tests.csv'}")



section("STEP 4 | Summary tables")

summary_cols = [
    "grid_position", "finishing_position",
    "positions_gained", "abs_position_change",
    "n_pit_stops", "first_pit_lap", "avg_pit_lap",
    "avg_pit_duration", "avg_pit_lane_duration",
    "n_stints", "avg_stint_length", "max_stint_length",
    "n_compounds_used",
    "temperature_mean", "humidity_mean",
    "precipitation_sum", "precipitation_max",
    "wind_speed_mean",
]
summary_cols = [c for c in summary_cols if c in analysis_df.columns]

summary_stats = analysis_df[summary_cols].describe().T
summary_stats["missing_count"] = analysis_df[summary_cols].isna().sum()
summary_stats["missing_pct"] = (
    summary_stats["missing_count"] / len(analysis_df) * 100
).round(2)
summary_stats.to_csv(TABLE_DIR / "summary_statistics.csv")

print("Summary statistics:")
display(summary_stats)

if "weather_class" in analysis_df.columns:
    weather_counts = (
        analysis_df["weather_class"].value_counts(dropna=False).to_frame("count")
    )
    weather_counts["pct"] = (weather_counts["count"] / len(analysis_df) * 100).round(2)
    weather_counts.to_csv(TABLE_DIR / "weather_class_counts.csv")
    print("\nWeather class counts:")
    display(weather_counts)

year_counts = analysis_df["year"].value_counts().sort_index().to_frame("count")
year_counts["unique_races"] = analysis_df.groupby("year")["session_key"].nunique()
year_counts.to_csv(TABLE_DIR / "year_counts.csv")
print("\nRows and races per year:")
display(year_counts)

if "dnf" in df_clean.columns and "weather_class" in df_clean.columns:
    dnf_by_weather = (
        df_clean.groupby("weather_class")
        .agg(n=("dnf", "size"), dnf_count=("dnf", "sum"))
    )
    dnf_by_weather["dnf_rate_pct"] = (
        dnf_by_weather["dnf_count"] / dnf_by_weather["n"] * 100
    ).round(2)
    dnf_by_weather.to_csv(TABLE_DIR / "dnf_rate_by_weather.csv")
    print("\nDNF rate by weather class (descriptive only):")
    display(dnf_by_weather)


#EDA plots
section("STEP 5 | Exploratory Data Analysis")

# 1. Grid position distribution
plt.figure(figsize=(8, 5))
plt.hist(analysis_df["grid_position"].dropna(), bins=20)
plt.xlabel("Grid Position")
plt.ylabel("Frequency")
plt.title("Distribution of Grid Position")
save_fig("hist_grid_position.png")

# 2. Finishing position distribution
plt.figure(figsize=(8, 5))
plt.hist(analysis_df["finishing_position"].dropna(), bins=20)
plt.xlabel("Finishing Position")
plt.ylabel("Frequency")
plt.title("Distribution of Finishing Position")
save_fig("hist_finishing_position.png")

# 3. Positions gained distribution
plt.figure(figsize=(8, 5))
plt.hist(analysis_df["positions_gained"].dropna(), bins=20)
plt.xlabel("Positions Gained")
plt.ylabel("Frequency")
plt.title("Distribution of Positions Gained")
save_fig("hist_positions_gained.png")

# 4. Grid vs finishing position
plot_df = analysis_df.dropna(subset=["grid_position", "finishing_position"]).copy()
max_pos = int(max(plot_df["grid_position"].max(), plot_df["finishing_position"].max()))
plt.figure(figsize=(7, 7))
plt.scatter(plot_df["grid_position"], plot_df["finishing_position"], alpha=0.6)
plt.plot([1, max_pos], [1, max_pos], linestyle="--")
plt.xlabel("Grid Position")
plt.ylabel("Finishing Position")
plt.title("Grid Position vs Finishing Position")
save_fig("scatter_grid_vs_finish.png")

# 5. Weather class vs absolute position change
weather_plot_df = analysis_df.dropna(
    subset=["weather_class", "abs_position_change"]
).copy()
valid_weather_order = [
    w for w in ["dry", "mixed", "wet"] if w in weather_plot_df["weather_class"].unique()
]

group_data = [
    weather_plot_df.loc[
        weather_plot_df["weather_class"] == w, "abs_position_change"
    ].dropna().values
    for w in valid_weather_order
]

if len(group_data) > 0:
    plt.figure(figsize=(8, 5))
    plt.boxplot(group_data, tick_labels=valid_weather_order)
    plt.xlabel("Weather Class")
    plt.ylabel("Absolute Position Change")
    plt.title("Absolute Position Change by Weather Class")
    save_fig("box_abs_position_change_by_weather.png")

# 6. Pit stops vs positions gained
pit_plot_df = analysis_df.dropna(subset=["n_pit_stops", "positions_gained"]).copy()
x = pit_plot_df["n_pit_stops"].values + np.random.normal(0, 0.04, len(pit_plot_df))
y = pit_plot_df["positions_gained"].values

plt.figure(figsize=(8, 5))
plt.scatter(x, y, alpha=0.6)
plt.xlabel("Number of Pit Stops")
plt.ylabel("Positions Gained")
plt.title("Pit Stops vs Positions Gained")
save_fig("scatter_pit_stops_vs_positions_gained.png")

# 7. First pit lap vs positions gained
fp_plot_df = analysis_df.dropna(subset=["first_pit_lap", "positions_gained"]).copy()
if not fp_plot_df.empty:
    plt.figure(figsize=(8, 5))
    plt.scatter(fp_plot_df["first_pit_lap"], fp_plot_df["positions_gained"], alpha=0.6)
    plt.xlabel("First Pit Lap")
    plt.ylabel("Positions Gained")
    plt.title("First Pit Lap vs Positions Gained")
    save_fig("scatter_first_pit_lap_vs_positions_gained.png")

# 8. Average stint length vs positions gained
stint_plot_df = analysis_df.dropna(
    subset=["avg_stint_length", "positions_gained"]
).copy()
if not stint_plot_df.empty:
    plt.figure(figsize=(8, 5))
    plt.scatter(
        stint_plot_df["avg_stint_length"],
        stint_plot_df["positions_gained"],
        alpha=0.6,
    )
    plt.xlabel("Average Stint Length")
    plt.ylabel("Positions Gained")
    plt.title("Average Stint Length vs Positions Gained")
    save_fig("scatter_avg_stint_length_vs_positions_gained.png")


section("STEP 6 | Correlation matrix")

corr_cols = [
    "grid_position", "finishing_position",
    "positions_gained", "abs_position_change",
    "n_pit_stops", "first_pit_lap", "avg_pit_lap",
    "avg_pit_duration", "avg_pit_lane_duration",
    "n_stints", "avg_stint_length", "max_stint_length",
    "n_compounds_used",
    "temperature_mean", "humidity_mean",
    "precipitation_sum", "precipitation_max",
    "wind_speed_mean",
]
corr_cols = [c for c in corr_cols if c in analysis_df.columns]

corr_matrix = analysis_df[corr_cols].corr(method="spearman", numeric_only=True)
corr_matrix.to_csv(TABLE_DIR / "correlation_matrix_spearman.csv")

print("Spearman correlation matrix:")
display(corr_matrix.round(3))

plt.figure(figsize=(11, 9))
im = plt.imshow(corr_matrix.values, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(im, label="Spearman correlation")
plt.xticks(range(len(corr_cols)), corr_cols, rotation=45, ha="right")
plt.yticks(range(len(corr_cols)), corr_cols)
plt.title("Spearman Correlation Matrix")
save_fig("correlation_matrix_heatmap.png")



section("STEP 7 | Hypothesis tests")

results_list = []

h1_df = analysis_df.dropna(subset=["grid_position", "finishing_position"]).copy()

h1_rho, h1_p_two = stats.spearmanr(h1_df["grid_position"], h1_df["finishing_position"])

if h1_rho > 0:
    h1_p_one = h1_p_two / 2
else:
    h1_p_one = 1 - h1_p_two / 2

h1_reject = (h1_p_one < 0.05) and (h1_rho > 0)

h1_result = pd.DataFrame({
    "hypothesis": ["H1"],
    "description": [
        "Starting grid position is positively associated with finishing position"
    ],
    "test": ["Spearman correlation (one-sided, positive)"],
    "n": [len(h1_df)],
    "spearman_rho": [h1_rho],
    "p_value_two_sided": [h1_p_two],
    "p_value_one_sided": [h1_p_one],
    "decision_0_05": ["Reject H0" if h1_reject else "Fail to reject H0"],
    "interpretation": [
        "Significant positive association between grid and finishing position"
        if h1_reject
        else "No significant positive association"
    ],
})
h1_result.to_csv(TABLE_DIR / "hypothesis_h1_result.csv", index=False)
display(h1_result)
results_list.append(h1_result)

h2_df = analysis_df.dropna(subset=["weather_group", "abs_position_change"]).copy()

group_dry = h2_df.loc[
    h2_df["weather_group"] == "dry", "abs_position_change"
].dropna()
group_wet = h2_df.loc[
    h2_df["weather_group"] == "wet_or_mixed", "abs_position_change"
].dropna()

h2_stat, h2_p = stats.mannwhitneyu(group_dry, group_wet, alternative="less")
h2_test = "Mann-Whitney U (one-sided, dry < wet_or_mixed)"

h2_effect = cohen_d(group_wet, group_dry)

h2_result = pd.DataFrame({
    "hypothesis": ["H2"],
    "description": [
        "Wet or mixed-weather races lead to greater absolute position changes than dry races"
    ],
    "test": [h2_test],
    "dry_n": [len(group_dry)],
    "wet_or_mixed_n": [len(group_wet)],
    "dry_mean_abs_change": [group_dry.mean()],
    "wet_or_mixed_mean_abs_change": [group_wet.mean()],
    "dry_median_abs_change": [group_dry.median()],
    "wet_or_mixed_median_abs_change": [group_wet.median()],
    "statistic": [h2_stat],
    "p_value_one_sided": [h2_p],
    "cohens_d_wet_minus_dry": [h2_effect],
    "decision_0_05": ["Reject H0" if h2_p < 0.05 else "Fail to reject H0"],
    "interpretation": [
        "Wet/mixed races show significantly greater position volatility than dry races"
        if h2_p < 0.05
        else "No significant evidence that wet/mixed races differ from dry races"
    ],
})
h2_result.to_csv(TABLE_DIR / "hypothesis_h2_result.csv", index=False)
display(h2_result)
results_list.append(h2_result)

strategy_vars = [
    "n_pit_stops",
    "first_pit_lap",
    "avg_pit_lap",
    "avg_pit_duration",
    "avg_pit_lane_duration",
    "avg_stint_length",
    "n_compounds_used",
]

h3_rows = []
for var in strategy_vars:
    if var not in analysis_df.columns:
        continue

    tmp = analysis_df.dropna(subset=[var, "positions_gained"]).copy()
    if len(tmp) < 3:
        continue

    rho, p_val = stats.spearmanr(tmp[var], tmp["positions_gained"])

    h3_rows.append({
        "hypothesis": "H3",
        "description": "Pit strategy variable associated with positions gained",
        "variable": var,
        "test": "Spearman correlation (two-sided)",
        "n": len(tmp),
        "spearman_rho": rho,
        "p_value": p_val,
    })

h3_results = pd.DataFrame(h3_rows)

if not h3_results.empty:
    reject, p_adj, _, _ = multipletests(
        h3_results["p_value"].values, alpha=0.05, method="holm"
    )
    h3_results["p_value_holm"] = p_adj
    h3_results["decision_holm_0_05"] = np.where(
        reject, "Reject H0", "Fail to reject H0"
    )
    h3_results["interpretation"] = np.where(
        reject,
        "Significant association after Holm correction",
        "Not significant after Holm correction",
    )

h3_results.to_csv(TABLE_DIR / "hypothesis_h3_univariate_results.csv", index=False)
print("H3 — univariate Spearman correlations with Holm correction:")
display(h3_results)

if not h3_results.empty:
    results_list.append(h3_results)

if results_list:
    all_hypothesis_results = pd.concat(
        results_list, ignore_index=True, sort=False
    )
    all_hypothesis_results.to_csv(
        TABLE_DIR / "all_hypothesis_results.csv", index=False
    )



section("STEP 8 | Regression model — H3 multivariate test")


regression_vars = [
    "finishing_position",
    "grid_position",
    "n_pit_stops",
    "first_pit_lap",
    "avg_stint_length",
    "n_compounds_used",
    "weather_class",
]
regression_vars = [c for c in regression_vars if c in analysis_df.columns]

reg_df = analysis_df[regression_vars].dropna().copy()

if "weather_class" in reg_df.columns:
    reg_df = reg_df[reg_df["weather_class"].isin(["dry", "mixed", "wet"])].copy()

print(f"Regression sample size: {len(reg_df)}")
display(reg_df.head())

model = None
if len(reg_df) >= 10:
    formula = (
        "finishing_position ~ grid_position + n_pit_stops + first_pit_lap "
        "+ avg_stint_length + n_compounds_used + C(weather_class)"
    )
    print(f"\nFormula: {formula}")

    model = smf.ols(formula=formula, data=reg_df).fit(cov_type="HC3")
    print(model.summary())

    reg_table = pd.DataFrame({
        "variable": model.params.index,
        "coefficient": model.params.values,
        "std_error": model.bse.values,
        "t_value": model.tvalues.values,
        "p_value": model.pvalues.values,
    })
    reg_table["significant_0_05"] = reg_table["p_value"] < 0.05
    reg_table.to_csv(
        TABLE_DIR / "regression_results_finishing_position.csv", index=False
    )

    print("\nRegression coefficients table:")
    display(reg_table)


    strategy_terms = [
        "n_pit_stops", "first_pit_lap",
        "avg_stint_length", "n_compounds_used",
    ]
    strategy_in_model = [v for v in strategy_terms if v in model.params.index]
    any_sig = any(model.pvalues[v] < 0.05 for v in strategy_in_model)

    h3_multivariate = pd.DataFrame({
        "hypothesis": ["H3 (multivariate)"],
        "description": [
            "At least one pit strategy variable is associated with race outcome "
            "after controlling for grid position and weather"
        ],
        "test": ["OLS regression (HC3 robust SE), DV=finishing_position"],
        "n": [len(reg_df)],
        "r_squared": [model.rsquared],
        "adj_r_squared": [model.rsquared_adj],
        "f_statistic": [model.fvalue],
        "f_pvalue": [model.f_pvalue],
        "strategy_vars_significant": [
            ", ".join(
                [v for v in strategy_in_model if model.pvalues[v] < 0.05]
            )
            or "none"
        ],
        "decision_0_05": ["Reject H0" if any_sig else "Fail to reject H0"],
        "interpretation": [
            "At least one strategy variable is significant controlling for grid and weather"
            if any_sig
            else "No strategy variable remains significant after controls"
        ],
    })
    h3_multivariate.to_csv(
        TABLE_DIR / "hypothesis_h3_multivariate_result.csv", index=False
    )
    print("\nH3 multivariate result:")
    display(h3_multivariate)
else:
    print("Not enough observations for regression.")


section("STEP 9 | Final summary")

print("Cleaned dataset summary:")
final_summary = pd.DataFrame({
    "metric": [
        "Rows in raw dataset",
        "Rows in cleaned dataset (incl. DNF)",
        "Rows in analysis dataset (excl. DNF/DNS/DSQ)",
        "Unique race sessions",
        "Unique years",
        "Missing grid_position",
        "Missing finishing_position",
        "Missing weather_class",
    ],
    "value": [
        len(df),
        len(df_clean),
        len(analysis_df),
        analysis_df["session_key"].nunique(),
        analysis_df["year"].nunique(),
        int(analysis_df["grid_position"].isna().sum()),
        int(analysis_df["finishing_position"].isna().sum()),
        int(analysis_df["weather_class"].isna().sum())
        if "weather_class" in analysis_df.columns
        else np.nan,
    ],
})
display(final_summary)
final_summary.to_csv(TABLE_DIR / "final_summary.csv", index=False)

print("\nKey interpretation notes:")
print("- H1 tests whether starting position is positively associated with finishing position (one-sided).")
print("- H2 tests whether wet/mixed races produce greater position volatility than dry races (one-sided).")
print("- H3 is tested both univariately (Spearman with Holm correction) and")
print("  multivariately (OLS regression controlling for grid position and weather).")
print("- DNF drivers are excluded from the analysis dataset because their finishing")
print("  positions reflect mechanical failure rather than race pace.")

if model is not None:
    print(f"\nRegression R-squared       : {model.rsquared:.4f}")
    print(f"Regression Adj. R-squared  : {model.rsquared_adj:.4f}")

print("\nSaved outputs:")
print(f"- Cleaned dataset        : {CLEAN_DATA_PATH}")
print(f"- Figures folder         : {FIG_DIR}")
print(f"- Tables folder          : {TABLE_DIR}")

section("DONE")